# 01 – Generate NDVI Rasters

## Purpose

This notebook generates NDVI rasters from Sentinel-2 **B04** and **B08** bands.

- **B04** = Red band
- **B08** = Near-infrared band

The output rasters are saved with clean date-based filenames.

```text
YYYY-MM-DD_NDVI.tif
```

## Workflow step

```text
Sentinel-2 B04 / B08 bands
        ↓
NDVI calculation
        ↓
NDVI_clean rasters
```

## 1. Import libraries and enable ArcPy Spatial Analyst

This workflow uses ArcPy raster algebra. The Spatial Analyst extension is required for `Raster` and `Float` operations.

In [ ]:
import os
from datetime import datetime

import arcpy
from arcpy.sa import *

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

## 2. Define input and output folders

Update these paths before running the notebook on another machine.

- `root_folder`: folder containing the original Sentinel-2 band rasters
- `output_folder`: folder where the generated NDVI rasters will be saved

In [ ]:
root_folder = r"path\to\input\NDVI"
output_folder = r"path\to\output\NDVI_clean"

## 3. Create output folder

If the output folder does not exist, it is created automatically.

In [ ]:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

print("Output folder ready:", output_folder)

## 4. Define helper functions

These helper functions make the processing logic easier to read and maintain.

- `is_raster_file()` checks whether a file is a GeoTIFF raster.
- `extract_date_from_filename()` extracts and validates the acquisition date from the first 10 characters of the filename.
- `find_b08_pair()` creates the expected B08 filename from the B04 filename.

In [ ]:
def is_raster_file(filename):
    """Return True if the file is a TIFF raster."""
    return filename.lower().endswith((".tif", ".tiff"))


def extract_date_from_filename(filename):
    """Extract and validate date from the first 10 characters of the filename."""
    date_str = filename[:10]
    datetime.strptime(date_str, "%Y-%m-%d")
    return date_str


def find_b08_pair(b04_filename):
    """Return the expected B08 filename based on the B04 filename."""
    return b04_filename.replace("B04", "B08")

## 5. Calculate NDVI for all matching B04 / B08 raster pairs

The notebook recursively searches the input folder for B04 rasters.  
For each B04 raster, it looks for the matching B08 raster in the same folder.

NDVI is calculated as:

```text
NDVI = (NIR - Red) / (NIR + Red)
```

In this workflow:

```text
Red = B04
NIR = B08
```

In [ ]:
processed_count = 0
skipped_count = 0

for subdir, dirs, files in os.walk(root_folder):

    # Select only B04 raster files from the current folder.
    b04_files = [
        f for f in files
        if "B04" in f and is_raster_file(f)
    ]

    for b04 in b04_files:

        b04_path = os.path.join(subdir, b04)

        # Find the matching B08 raster.
        b08 = find_b08_pair(b04)
        b08_path = os.path.join(subdir, b08)

        if not os.path.exists(b08_path):
            print("Missing B08 pair:", b08)
            skipped_count += 1
            continue

        try:
            # Extract acquisition date from filename.
            date_str = extract_date_from_filename(b04)

        except ValueError:
            print("Invalid date format:", b04)
            skipped_count += 1
            continue

        print("Processing:", b04)

        red = Raster(b04_path)
        nir = Raster(b08_path)

        # Calculate NDVI using floating point raster algebra.
        ndvi = (Float(nir) - Float(red)) / (Float(nir) + Float(red))

        out_name = f"{date_str}_NDVI.tif"
        out_path = os.path.join(output_folder, out_name)

        ndvi.save(out_path)

        print("Saved:", out_name)
        processed_count += 1

print("\nNDVI generation finished.")
print("Processed rasters:", processed_count)
print("Skipped rasters:", skipped_count)

## 6. Output

The generated NDVI rasters are saved in:

```text
NDVI_clean
```

Expected output filename example:

```text
2020-05-18_NDVI.tif
```

These rasters are used as input for the next notebook:

```text
02_create_seasonal_mosaics.ipynb
```